In [1]:
import sys
import os
sys.path.append('/root/LLMs/GLM-4-Voice')

import torch
from flow_inference import AudioDecoder
import os.path


flow_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-decoder"
tokenizer_path="/root/LLMs/GLM-4-Voice/THUDM/glm-4-voice-tokenizer"

flow_config = os.path.join(flow_path, "config.yaml")
flow_checkpoint = os.path.join(flow_path, "flow.pt")
hift_checkpoint = os.path.join(flow_path, "hift.pt")
device = "cuda"
audio_decoder: AudioDecoder = None
whisper_model, feature_extractor = None, None

sys.path.insert(0, "/root/LLMs/GLM-4-Voice/cosyvoice")
sys.path.insert(0, "/root/LLMs/CosyVoice/third_party/Matcha-TTS")

In [2]:
import re

def extract_numbers_from_string(s):
    # 使用正则表达式找到所有符合数字模式的序列，包括负数和小数
    number_pattern = r'-?\d+\.?\d*'
    matches = re.findall(number_pattern, s)
    
    # 将找到的字符串格式的数字转换为适当的数值类型（整数或浮点数）
    numbers = []
    for match in matches:
        try:
            # 如果字符串包含小数点，则转换为浮点数
            if '.' in match:
                numbers.append(float(match))
            else:
                # 否则转换为整数
                numbers.append(int(match))
        except ValueError:
            # 如果转换失败，忽略该匹配项
            continue
    
    return numbers

In [3]:
audio_tokens = "北京啊，那可是个充满活力和魅力的地方！<|audio_10815|><|audio_14121|><|audio_14268|><|audio_15298|><|audio_7506|><|audio_15400|><|audio_13364|><|audio_14248|><|audio_14445|><|audio_15428|><|audio_10640|><|audio_557|><|audio_3322|><|audio_10973|><|audio_2968|><|audio_13089|><|audio_1213|><|audio_12153|><|audio_8995|><|audio_5838|><|audio_13549|><|audio_14720|><|audio_4623|><|audio_6648|><|audio_8160|><|audio_10148|>无论是历史悠久的前门、故宫，还是现代化的三里屯、<|audio_1329|><|audio_10023|><|audio_3148|><|audio_8336|><|audio_13255|><|audio_5422|><|audio_6353|><|audio_13933|><|audio_5554|><|audio_4781|><|audio_6237|><|audio_11993|><|audio_14934|><|audio_1733|><|audio_1504|><|audio_6914|><|audio_13800|><|audio_12714|><|audio_16007|><|audio_12041|><|audio_12890|><|audio_5296|><|audio_10420|><|audio_10693|><|audio_5900|><|audio_8227|>国贸，都能让你流连忘返。而且北京的美食<|audio_8790|><|audio_5889|><|audio_11346|><|audio_10813|><|audio_362|><|audio_14166|><|audio_12463|><|audio_692|><|audio_12683|><|audio_6237|><|audio_624|><|audio_15726|><|audio_13558|><|audio_7856|><|audio_11684|><|audio_13664|><|audio_15528|><|audio_8206|><|audio_15287|><|audio_12959|><|audio_4230|><|audio_8884|><|audio_11734|><|audio_13307|><|audio_9614|><|audio_13612|>也是全国各地的风味都有，绝对能让你大饱口福<|audio_11495|><|audio_11024|><|audio_1769|><|audio_15069|><|audio_3822|><|audio_9716|><|audio_4401|><|audio_317|><|audio_13089|><|audio_9435|><|audio_10391|><|audio_9947|><|audio_1511|><|audio_4804|><|audio_15775|><|audio_6131|><|audio_4421|><|audio_1474|><|audio_5674|><|audio_11342|><|audio_12594|><|audio_3032|><|audio_14901|><|audio_14399|><|audio_12714|><|audio_15526|>！<|audio_14458|><|audio_9102|><|audio_12463|><|audio_4113|><|audio_7553|><|audio_13964|><|audio_15569|><|audio_5200|><|audio_485|><|audio_13176|><|audio_4336|><|audio_4357|><|audio_1392|><|audio_7746|><|audio_14593|><|audio_1322|><|audio_13627|><|audio_10036|><|audio_15470|><|audio_12714|><|audio_16297|><|audio_5913|><|audio_11783|><|audio_12123|><|audio_1502|><|audio_11585|><|audio_9546|><|audio_1871|><|audio_5764|><|audio_15935|><|audio_10660|><|audio_12447|><|audio_292|><|audio_12057|><|audio_8886|><|audio_10293|><|audio_120|><|audio_16059|><|audio_11151|><|audio_13706|><|audio_3793|><|audio_9306|><|audio_3412|><|audio_2771|><|audio_9653|><|audio_3954|><|audio_7260|><|audio_15662|><|audio_13737|><|audio_12112|><|audio_3374|><|audio_14720|><|audio_5389|><|audio_4143|><|audio_41|><|audio_11806|><|audio_8809|><|audio_14383|><|audio_6317|><|audio_9247|><|audio_7384|><|audio_1789|><|audio_15848|><|audio_6105|><|audio_4726|><|audio_13162|><|audio_11002|><|audio_15444|><|audio_3260|><|audio_379|><|audio_2372|><|audio_15120|><|audio_8406|><|audio_14050|><|audio_4225|><|audio_1280|><|audio_9875|><|audio_5354|><|audio_10192|><|audio_1051|><|audio_12041|><|audio_15724|>"

In [10]:
audio_tokens = extract_numbers_from_string(audio_tokens)

NameError: name 'extract_numbers_from_string' is not defined

In [11]:
# Flow & Hift
audio_decoder = AudioDecoder(config_path=flow_config,
                             flow_ckpt_path=flow_checkpoint,
                             hift_ckpt_path=hift_checkpoint,
                             device=device,)

import uuid
# 生成唯一标识符
this_uuid = str(uuid.uuid4())

# 生成语音和梅尔特征
with torch.no_grad():
    # 转换为张量
    tts_token = torch.tensor(audio_tokens, device=device).unsqueeze(0)
    tts_token
    tts_speech, tts_mel = audio_decoder.token2wav(
        tts_token,
        # prompt_token = tts_token,
        uuid=this_uuid
    )

tts_speech.shape

torch.Size([1, 77568])

In [12]:
import IPython.display as ipd

# 假设 audio_data 是你的音频数据，sr 是采样率
# audio_data = np.array(new_dataset['mp3'][0]['array'])
sr = 22050 
tts_speech = tts_speech.cpu()
# 在 Jupyter Notebook 中播放音频
ipd.Audio(tts_speech, rate=sr)